In [ ]:
# Cell 1: Title, purpose, and setup
import os
os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, SVG, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.design_model import (
    compute_balance_residuals,
    compute_objective_decomposition,
    compute_participant_economics,
    extract_design_solution,
    extract_node_product_prices,
    find_minimum_price_for_target,
    solve_design_problem,
    solve_fixed_design_management_problem,
    validate_design_state,
)
from src.hw2_plastic_waste import (
    DEFAULT_HOMEWORK_PDF,
    ETHYLENE,
    HW2_STRUCTURED_PROSE,
    PLASTIC_WASTE,
    PYROLYSIS_OIL,
    build_hw2_plastic_waste_semantic_plan,
    build_hw2_plastic_waste_state,
    total_accepted_ethylene,
    total_available_plastic_waste,
    total_recycled_plastic_waste,
)
from src.llm_problem_interpreter import (
    build_problem_artifacts_from_semantic_plan,
    build_state_from_semantic_plan,
    interpret_problem_from_text,
    summarize_problem_state,
)
from src.network_graph import build_problem_graph_spec, build_solution_graph_spec
from src.network_visualizer import GRAPHVIZ_INSTALL_MESSAGE, render_graphviz, render_mermaid
from src.solver_results import SolverResults
from src.validator import validate_state

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

def show_title(text, body=None):
    display(Markdown(f"## {text}" + (f"\n\n{body}" if body else "")))

def show_json(obj):
    display(Markdown("```json\n" + json.dumps(obj, indent=2, default=str) + "\n```"))

def df(records, columns=None):
    return pd.DataFrame(records, columns=columns)

show_title(
    "ChatbotLP HW#2 Plastic Waste Showcase",
    "This notebook demonstrates the framework pipeline on the plastic waste processing homework one question at a time: prose interpretation, `ProblemState`, validation, MILP design, fixed-design LP pricing, network visualization, balance checks, and a willingness-to-pay sweep."
)
print(f"Repository root: {REPO_ROOT}")
print(f"LLM provider: {os.environ.get('LLM_PROVIDER')}")
print(f"Gemini model: {os.environ.get('GEMINI_MODEL')}")
print(f"Gemini API key configured: {bool(os.environ.get('GEMINI_API_KEY'))}")

In [ ]:
# Cell 2: Load or paste homework problem statement
show_title("Problem Statement")

pdf_candidates = [
    DEFAULT_HOMEWORK_PDF,
    REPO_ROOT / "Benchmarks" / "hw2_plastic_waste" / "hw2_supply_chains_spring2022.pdf",
]
existing_pdf = next((path for path in pdf_candidates if path.exists()), None)
if existing_pdf is not None:
    print(f"Homework PDF found: {existing_pdf.relative_to(REPO_ROOT)}")
else:
    print("Homework PDF not found. Place it at Benchmarks/hw2_plastic_waste/hw2_supply_chains_spring2022.pdf")

problem_text = """# HW#2

Due: February 17th, 2022

CBE450: Process Design - Spring 2022

Department of Chemical and Biological Engineering, University of Wisconsin-Madison

## Supply Chain for Plastic Waste Processing (100 pts)

Plastic waste affects the environment in many ways; it is estimated that 3.5 million tons of plastic waste are generated every year in the Upper Midwest region of the United States. Fortunately, there is a new promising approach for dealing with plastic waste in a scalable manner. This pathway uses thermochemical technologies (pyrolysis and steam cracking) to break plastic and recover value-added chemicals. As shown in Figure 1, pyrolysis converts the plastic waste into a pyrolysis oil, while steam cracking produces ethylene from the pyrolysis oil.

Figure 1: Schematic of plastic waste processing pathway

We would like to design and operate a supply chain that collects and processes waste and obtains value-added products (ethylene). You are given the following data:

- Supply Information: The goal is to recycle all the plastic waste generated in Madison (MAD) and Milwaukee (MKE). The total plastic waste generated in MAD and MKE is 33100, 77300 ton/yr, respectively. The supply cost for the plastic waste from these cities is 0 $/ton (plastic waste is offered for free).

- Demand Information: Each city has a consumer of ethylene and we wish to satisfy the demands of such consumers. The demand capacities for ethylene are both 100,000 tons/yr and the consumers are both willing to pay 1050 $/ton.

- Transportation Information: The transportation cost for the three products (plastic, pyrolysis oil, and ethylene) are the same. The cost to move one ton of product from MAD to MKE (and from MKE to MAD) is 10$/ton. The transportation capacity for all products is 100,000 ton/yr.

- Technology Information:
  - You have the possibility of installing a pyrolysis process in MAD and another one in MKE; you also have the possibility of installing a steam cracking process in MAD and another one in MKE.
  - For the pyrolysis process, 0.7 ton of pyrolysis oil can be produced from breaking one ton of plastic waste. The operational cost of pyrolyzing one ton of plastic waste is 14 $. The maximum capacity of a single pyrolysis facility is 100,000 tons of plastic waste per year. The investment cost for this technology is 2,000,000 $/yr (annualized investment).
  - The steam cracking process converts one ton of pyrolysis oil into 0.25 ton of ethylene. The operational cost of steam cracking is 71$ per ton of pyrolysis oil. A single steam cracking facility process 100,000 ton of pyrolysis oil each year. The investment cost for this technology is 2,800,000 $/yr (annualized investment).
"""

display(Markdown(problem_text.replace("$", "\\$")))

In [ ]:
# Cell 3: Natural prose to semantic plan
show_title("Natural Prose to Semantic Plan")

use_llm = bool(os.environ.get("GEMINI_API_KEY"))
evaluation_mode = "live_llm" if use_llm else "deterministic_fixture"
interpretation_mode = evaluation_mode
semantic_plan = None
llm_error = None

if use_llm:
    try:
        artifacts = interpret_problem_from_text(problem_text)
        semantic_plan = artifacts["semantic_plan"]
    except Exception as exc:
        llm_error = exc
        print(f"Live LLM evaluation failed: {exc}")
        raise
else:
    semantic_plan = build_hw2_plastic_waste_semantic_plan()

print(f"evaluation_mode = {evaluation_mode!r}")
print(f"Interpretation mode: {interpretation_mode}")
if llm_error is not None:
    print(f"LLM interpretation error: {llm_error}")

show_json(semantic_plan)

In [ ]:
# Cell 4: ProblemState construction and validation
show_title("ProblemState Construction and Validation")

state = build_state_from_semantic_plan(semantic_plan)
state_summary = summarize_problem_state(state)
base_validation = validate_state(state)
design_validation = validate_design_state(state)

missing_demo_state = build_hw2_plastic_waste_state(missing_investment_cost=True)
missing_design_validation = validate_design_state(missing_demo_state)

print("ProblemState summary")
show_json(state_summary)

print("Base validation")
show_json({
    "solver_ready": base_validation["solver_ready"],
    "missing_parameters": base_validation["missing_parameters"],
    "invalid_references": base_validation["invalid_references"],
    "incomplete_technologies": base_validation["incomplete_technologies"],
})

print("Fixed-charge design validation")
show_json({
    "solver_ready": design_validation["solver_ready"],
    "missing_parameters": design_validation["missing_parameters"],
})

print("Missing-value discipline check: removing investment costs leaves fixed_cost as None and blocks design solving.")
show_json({
    "solver_ready": missing_design_validation["solver_ready"],
    "missing_parameters": missing_design_validation["missing_parameters"],
    "fixed_cost_values": {technology.id: technology.fixed_cost for technology in missing_demo_state.technologies},
})

In [ ]:
# Cell 5: Q1 — Sets of participants
show_title("Q1 — Sets of Participants")

nodes_table = df([{"node": node.id, "name": node.name} for node in state.nodes])
products_table = df([{"product": product.id, "name": product.name} for product in state.products])
suppliers_table = df([{"supplier": s.id, "node": s.node, "product": s.product, "capacity_ton_per_year": s.capacity} for s in state.suppliers])
consumers_table = df([{"consumer": c.id, "node": c.node, "product": c.product, "capacity_ton_per_year": c.capacity} for c in state.consumers])
transport_table = df([{"link": t.id, "origin": t.origin, "destination": t.destination, "product": t.product} for t in state.transport_links])
technology_table = df([{"technology": k.id, "candidate_location": k.node} for k in state.technologies])

for label, table in [
    ("Nodes/locations", nodes_table),
    ("Products", products_table),
    ("Suppliers", suppliers_table),
    ("Consumers", consumers_table),
    ("Transport links", transport_table),
    ("Technologies and candidate locations", technology_table),
]:
    display(Markdown(f"**{label}**"))
    display(table)

In [ ]:
# Cell 6: Q2 — Data
show_title("Q2 — Data")

bid_by_owner = {(bid.owner_type, bid.owner_id): bid for bid in state.bids}

supply_data = []
for supplier in state.suppliers:
    bid = bid_by_owner[("supplier", supplier.id)]
    supply_data.append({
        "supplier": supplier.id,
        "node": supplier.node,
        "product": supplier.product,
        "capacity": supplier.capacity,
        "capacity_units": "ton/year",
        "supply_cost": bid.price,
        "cost_units": "$/ton",
    })

demand_data = []
for consumer in state.consumers:
    bid = bid_by_owner[("consumer", consumer.id)]
    demand_data.append({
        "consumer": consumer.id,
        "node": consumer.node,
        "product": consumer.product,
        "capacity": consumer.capacity,
        "capacity_units": "ton/year",
        "willingness_to_pay": bid.price,
        "price_units": "$/ton",
    })

transport_data = [
    {
        "link": link.id,
        "origin": link.origin,
        "destination": link.destination,
        "product": link.product,
        "capacity": link.capacity,
        "capacity_units": "ton/year",
        "cost": link.cost,
        "cost_units": "$/ton",
    }
    for link in state.transport_links
]

technology_data = [
    {
        "technology": technology.id,
        "node": technology.node,
        "capacity": technology.capacity,
        "capacity_units": "input ton/year",
        "operating_cost": technology.cost,
        "operating_cost_units": "$/input ton",
        "investment_cost": technology.fixed_cost,
        "investment_cost_units": "$/year",
        "yield_coefficients": technology.yield_coefficients,
    }
    for technology in state.technologies
]

for label, table in [
    ("Supply data", df(supply_data)),
    ("Demand data", df(demand_data)),
    ("Transport data", df(transport_data)),
    ("Technology data", df(technology_data)),
]:
    display(Markdown(f"**{label}**"))
    display(table)

In [ ]:
# Cell 7: Q3 — Problem graph
show_title("Q3 — Problem Graph")

problem_graph = build_problem_graph_spec(state)
mermaid_problem = render_mermaid(problem_graph).replace("missing", "?")
print(mermaid_problem)

try:
    problem_svg = render_graphviz(problem_graph, str(REPO_ROOT / "notebooks" / "hw2_problem_graph.svg"))
    display(SVG(filename=problem_svg))
except RuntimeError as exc:
    if str(exc) == GRAPHVIZ_INSTALL_MESSAGE:
        print(GRAPHVIZ_INSTALL_MESSAGE)
    else:
        raise

In [ ]:
# Cell 8: Q4 — Design optimization
show_title("Q4 — Design Optimization")

if not design_validation["solver_ready"]:
    raise ValueError(f"Design model is not solver-ready: {design_validation['missing_parameters']}")

design_result = solve_design_problem(state, solver_name="glpk")
design_summary = extract_design_solution(state, design_result)

print(design_result.message)
print(f"Objective value: {design_result.objective_value:,.2f} $/year" if design_result.objective_value is not None else "Objective unavailable")

selected_rows = [
    {
        "technology": technology.id,
        "node": technology.node,
        "installed_y": design_summary["technology_installations"].get(technology.id, 0.0),
        "activity": design_summary["technology_activities"].get(technology.id, 0.0),
        "capacity": technology.capacity,
    }
    for technology in state.technologies
]
flow_rows = [
    {
        "link": link.id,
        "origin": link.origin,
        "destination": link.destination,
        "product": link.product,
        "flow": design_summary["transport_flows"].get(link.id, 0.0),
        "capacity": link.capacity,
    }
    for link in state.transport_links
]
accepted_supply_rows = [
    {"supplier": supplier, "accepted_supply": quantity}
    for supplier, quantity in design_summary["accepted_supply"].items()
]
accepted_demand_rows = [
    {"consumer": consumer, "accepted_demand": quantity}
    for consumer, quantity in design_summary["accepted_demand"].items()
]

display(Markdown("**Selected technologies and activities**"))
display(df(selected_rows))
display(Markdown("**Accepted supply**"))
display(df(accepted_supply_rows))
display(Markdown("**Accepted demand**"))
display(df(accepted_demand_rows))
display(Markdown("**Transport flows**"))
display(df(flow_rows))

In [ ]:
# Cell 9: Q5 — Solution graph and balance verification
show_title("Q5 — Solution Graph and Balance Verification")

solver_results = SolverResults.from_solve_result(design_result, state)
solution_graph = build_solution_graph_spec(state, solver_results)
mermaid_solution = render_mermaid(solution_graph).replace("missing", "?")
print(mermaid_solution)

try:
    solution_svg = render_graphviz(solution_graph, str(REPO_ROOT / "notebooks" / "hw2_solution_graph.svg"))
    display(SVG(filename=solution_svg))
except RuntimeError as exc:
    if str(exc) == GRAPHVIZ_INSTALL_MESSAGE:
        print(GRAPHVIZ_INSTALL_MESSAGE)
    else:
        raise

balance_rows = compute_balance_residuals(state, design_result)
balance_table = df(balance_rows)
display(balance_table)
print(f"All balances hold within tolerance: {bool(balance_table['holds'].all())}")

In [ ]:
# Cell 10: Q6 — Fixed-design management problem
show_title("Q6 — Fixed-Design Management Problem")

installed_design = design_summary["selected_technologies"]
management_result = solve_fixed_design_management_problem(state, installed_design, solver_name="glpk")
management_summary = extract_design_solution(state, management_result)
node_product_prices = extract_node_product_prices(management_result)

print(management_result.message)
if node_product_prices:
    price_rows = [
        {"node": key.split(":", 1)[0], "product": key.split(":", 1)[1], "price": value, "units": "$/ton"}
        for key, value in sorted(node_product_prices.items())
    ]
    display(Markdown("**Node-product prices from fixed-design LP duals**"))
    display(df(price_rows))
else:
    print("Dual prices are unavailable from this solver/model combination.")

economics = compute_participant_economics(state, management_result, node_product_prices, installed_design) if node_product_prices else {}
for label, key in [
    ("Supplier revenues/profits", "suppliers"),
    ("Consumer payments/surplus", "consumers"),
    ("Transporter revenues/costs/profits", "transporters"),
    ("Technology costs/revenues/profits", "technologies"),
]:
    if economics:
        display(Markdown(f"**{label}**"))
        display(df(economics[key]))

In [ ]:
# Cell 11: Q7 — Total profit
show_title("Q7 — Total Profit")

decomposition = compute_objective_decomposition(state, design_result)
decomp_table = df([
    {"component": key, "value_$per_year": value}
    for key, value in decomposition.items()
])
display(decomp_table)
print(
    "Total profit equals demand revenue minus supplier cost, transport cost, "
    "operating cost, and investment cost."
)

In [ ]:
# Cell 12: Q8 — Actual value of plastic waste
show_title("Q8 — Actual Value of Plastic Waste")

plastic_price_rows = []
for node in state.node_ids():
    plastic_price_rows.append({
        "node": node,
        "plastic_waste_price": node_product_prices.get(f"{node}:{PLASTIC_WASTE}"),
        "units": "$/ton",
    })
display(df(plastic_price_rows))

supplier_profit_rows = economics.get("suppliers", []) if node_product_prices else []
display(df(supplier_profit_rows))

mad_price = node_product_prices.get(f"MAD:{PLASTIC_WASTE}")
mke_price = node_product_prices.get(f"MKE:{PLASTIC_WASTE}")
if mad_price is not None and mke_price is not None:
    display(Markdown(
        f"Plastic waste has location-dependent value in the fixed design: MAD price = {mad_price:.2f} $/ton and MKE price = {mke_price:.2f} $/ton. "
        "The MAD price is lower because MAD plastic must pay transport to reach the active MKE processing chain; the MKE waste is locally scarce at the selected facility."
    ))
else:
    display(Markdown("Plastic-waste prices could not be recovered because dual prices were unavailable."))

In [ ]:
# Cell 13: Q9 — Does the design recycle all plastic waste?
show_title("Q9 — Does the Designed Supply Chain Recycle All Plastic Waste?")

total_available = total_available_plastic_waste(state)
total_recycled = total_recycled_plastic_waste(state, design_result)
total_unprocessed = total_available - total_recycled
ethylene_accepted = total_accepted_ethylene(state, design_result)

recycling_table = df([
    {"metric": "available plastic waste", "value": total_available, "units": "ton/year"},
    {"metric": "accepted/recycled plastic waste", "value": total_recycled, "units": "ton/year"},
    {"metric": "unprocessed plastic waste", "value": total_unprocessed, "units": "ton/year"},
    {"metric": "accepted ethylene demand", "value": ethylene_accepted, "units": "ton/year"},
])
display(recycling_table)

display(Markdown(
    "The design does not recycle all available plastic waste at 1050 $/ton ethylene WTP. "
    "The selected MKE pyrolysis unit reaches its 100000 ton/year plastic capacity. "
    "Recycling the remaining 10400 ton/year would require additional fixed investment, "
    "and the solver-backed objective shows that extra investment is not justified at this WTP."
))

In [ ]:
# Cell 14: Q10 — Minimum ethylene WTP for full recycling
show_title("Q10 — Minimum Ethylene WTP for Full Recycling")

threshold_result = find_minimum_price_for_target(
    state,
    product_id=ETHYLENE,
    target_quantity=total_available,
    quantity_getter=total_recycled_plastic_waste,
    low=500.0,
    high=2500.0,
    coarse_steps=17,
    bisection_iterations=20,
    solver_name="glpk",
)

coarse_rows = threshold_result["coarse_rows"]
coarse_table = df([
    {
        "ethylene_wtp": row["price"],
        "recycled_plastic": row["quantity"],
        "objective": row["objective_value"],
        "selected_technologies": ", ".join(row["selected_technologies"]),
    }
    for row in coarse_rows
])
display(coarse_table)

threshold = threshold_result["threshold"]
print(f"Threshold WTP: {threshold:.6f} $/ton" if threshold is not None else threshold_result["message"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(coarse_table["ethylene_wtp"], coarse_table["recycled_plastic"], marker="o")
ax.axhline(total_available, color="black", linestyle="--", linewidth=1, label="available plastic")
if threshold is not None:
    ax.axvline(threshold, color="tab:red", linestyle=":", linewidth=1.5, label=f"threshold {threshold:.2f}")
ax.set_xlabel("Ethylene willingness to pay ($/ton)")
ax.set_ylabel("Recycled plastic waste (ton/year)")
ax.set_title("Full-recycling threshold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

if threshold is not None:
    threshold_state = threshold_result["threshold_state"]
    threshold_design_result = threshold_result["threshold_result"]
    threshold_summary = extract_design_solution(threshold_state, threshold_design_result)
    threshold_management = solve_fixed_design_management_problem(
        threshold_state,
        threshold_summary["selected_technologies"],
        solver_name="glpk",
    )
    threshold_prices = extract_node_product_prices(threshold_management)
    display(Markdown("**Threshold design**"))
    display(df([
        {
            "technology": technology,
            "installed": installed,
            "activity": threshold_summary["technology_activities"].get(technology),
        }
        for technology, installed in sorted(threshold_summary["technology_installations"].items())
    ]))
    display(Markdown("**Plastic waste value under full-recycling threshold design**"))
    display(df([
        {"node": node, "plastic_waste_price": threshold_prices.get(f"{node}:{PLASTIC_WASTE}"), "units": "$/ton"}
        for node in threshold_state.node_ids()
    ]))

In [ ]:
# Cell 15: Summary for advisor
show_title("Summary for Advisor")

summary_points = [
    "Natural prose can be converted to a semantic plan in live LLM mode, or shown with an offline deterministic fixture demonstration when no Gemini key is configured.",
    "The central `ProblemState` carries explicit suppliers, consumers, transport links, products, and transformation technologies.",
    "Validation distinguishes missing values from explicit zeros; missing investment costs remain `None` and block design solving.",
    "The fixed-charge MILP selects technology locations and product allocations without hardcoding the plastic-waste homework into the core builder.",
    "The solution graph overlays active flows and selected technologies on the problem graph.",
    "Balance residuals are computed for every node-product pair and are solver-grounded.",
    "The fixed-design LP recovers node-product dual prices for pricing and participant economics when GLPK duals are available.",
    "The WTP sweep performs a solver-backed what-if analysis and estimates the full-recycling threshold.",
]
limitations = [
    "The live LLM path depends on a configured Gemini API key and fails visibly if that live call fails; the no-key path is an offline deterministic fixture demonstration.",
    "Dual prices are extracted from the fixed-design LP, not from the MILP, because MILP duals are not economically meaningful in the same way.",
    "Participant profit accounting uses the recovered nodal prices and the model's current market-clearing conventions; interpretation should stay tied to those equations.",
    "Graphviz SVG rendering is optional; Mermaid text is always rendered.",
]

display(Markdown("**Capabilities demonstrated**\n" + "\n".join(f"- {point}" for point in summary_points)))
display(Markdown("**Current limitations**\n" + "\n".join(f"- {item}" for item in limitations)))